In [ ]:
# LinearRegression

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Training data
X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5]
])

y = np.array([
    50,
    70,
    90,
    110,
    130
])

# Create model
model = LinearRegression()

# Train model
model.fit(X, y)

# Make prediction
prediction = model.predict([[6]])

print("Predicted score:", prediction[0])

In [ ]:
print("Coefficient:", model.coef_)
print("Intercept:", model.intercept_)

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([
    [2, 6, 50],
    [3, 7, 55],
    [4, 7, 60],
    [5, 8, 65],
    [6, 8, 70],
    [7, 8, 75]
])

y = np.array([
    55,
    60,
    65,
    70,
    75,
    80
])

model = LinearRegression()

model.fit(X, y)

new_student = [[8, 8, 80]]

prediction = model.predict(new_student)

print("Predicted score:", prediction[0])

print("Coefficient:", model.coef_)
print("Intercept:", model.intercept_)

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([
    [1],
    [2],
    [3],
    [5],
    [7],
    [10]
])

y = np.array([
    400000,
    500000,
    600000,
    800000,
    1100000,
    1500000
])

# Create model
model = LinearRegression()

# Train model
model.fit(X, y)

# Make prediction
prediction = model.predict([[12]])

print("Predicted salary:", prediction[0])

print("Coefficient:", model.coef_)
print("Intercept:", model.intercept_)

In [ ]:
# LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

X = np.array([[0.15], [0.55], [0.30], [0.70]])
y = np.array([0, 1, 0, 1])  # 0 = no default, 1 = default

model = LogisticRegression()
model.fit(X, y)

print("Probability of default at DTI=0.60:", model.predict_proba([[0.60]]))
print("Predicted class:", model.predict([[0.60]]))

In [ ]:
"""
K-Means Clustering — worked example
Segmenting customers by order frequency and average order value.

Requires: pip install scikit-learn matplotlib pandas numpy
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------
# 1. Create sample data (in practice: pd.read_csv("customers.csv"))
# ---------------------------------------------------------------
np.random.seed(42)

cluster_a = np.random.normal(loc=[5, 23], scale=[1.2, 4], size=(40, 2))   # frequent, low value
cluster_b = np.random.normal(loc=[3, 87], scale=[0.8, 6], size=(30, 2))   # rare, high value
cluster_c = np.random.normal(loc=[9, 90], scale=[1.0, 6], size=(30, 2))   # frequent, high value

data = np.vstack([cluster_a, cluster_b, cluster_c])
df = pd.DataFrame(data, columns=["order_frequency", "avg_order_value"])

print(df.describe())

# ---------------------------------------------------------------
# 2. Scale features — K-Means uses distance, so unscaled features
#    with big ranges (e.g. dollars vs. counts) will dominate.
# ---------------------------------------------------------------
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

# ---------------------------------------------------------------
# 3. Elbow method — try K from 1 to 8, plot inertia
# ---------------------------------------------------------------
inertias = []
k_range = range(1, 9)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(scaled_data)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("Elbow Method for choosing K")
plt.tight_layout()
plt.savefig("elbow_plot.png", dpi=150)
plt.close()
print("Saved elbow_plot.png — look for the 'bend' (here, K=3).")

# ---------------------------------------------------------------
# 4. Fit final model with chosen K
# ---------------------------------------------------------------
K = 3
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
df["cluster"] = kmeans.fit_predict(scaled_data)

# Centroids need to be un-scaled back to original units for interpretation
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)

print("\nCluster centroids (original units):")
for i, (freq, value) in enumerate(centroids_original):
    print(f"  Cluster {i}: order_frequency={freq:.1f}, avg_order_value=${value:.0f}")

print("\nCluster sizes:")
print(df["cluster"].value_counts().sort_index())

# ---------------------------------------------------------------
# 5. Visualize the clusters
# ---------------------------------------------------------------
plt.figure(figsize=(6, 5))
colors = ["#2a78d6", "#eb6834", "#1baf7a"]

for cluster_id in range(K):
    subset = df[df["cluster"] == cluster_id]
    plt.scatter(
        subset["order_frequency"],
        subset["avg_order_value"],
        c=colors[cluster_id],
        label=f"Cluster {cluster_id}",
        alpha=0.7,
    )

plt.scatter(
    centroids_original[:, 0],
    centroids_original[:, 1],
    c="black",
    marker="X",
    s=200,
    label="Centroids",
)

plt.xlabel("Order frequency (per month)")
plt.ylabel("Average order value ($)")
plt.title("Customer segments (K-Means, K=3)")
plt.legend()
plt.tight_layout()
plt.savefig("kmeans_clusters.png", dpi=150)
plt.close()
print("Saved kmeans_clusters.png")

# ---------------------------------------------------------------
# 6. Business labels (manual step — inspect centroids and name them)
# ---------------------------------------------------------------
labels = {
    0: "Frequent, small-ticket buyers",
    1: "Occasional big spenders",
    2: "VIP / key accounts",
}
df["segment_label"] = df["cluster"].map(labels)
print("\nSample labeled rows:")
print(df.head())